# Module 0.4 — Tokenization Deep Dive
**DeskMate SLM Curriculum · Phase 0 · Notebook 01**

Run top-to-bottom on **Google Colab** or **Kaggle** (CPU is fine — no GPU needed).

By the end you will have:
- Explored GPT-2's tokenizer on support-desk text
- Trained a domain Byte-level BPE tokenizer from scratch
- A side-by-side comparison showing how token counts differ

Read `0.4_tokenization.md` first for the theory behind each step.

---


## Step 0 — Install Dependencies

In [ ]:
%%capture
!pip install -q tokenizers==0.19.1 transformers==4.44.0 matplotlib==3.9.0


In [ ]:
# Verify
import tokenizers, transformers, matplotlib
print(f"tokenizers : {tokenizers.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"matplotlib : {matplotlib.__version__}")


## Step 1 — Seed & Paths

In [ ]:
import random, os, pathlib
random.seed(42)

# Detect runtime and set project root
try:
    import google.colab
    RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets
        RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

MODELS_DIR = PROJECT_ROOT / "models" / "tokenizer"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Runtime     : {RUNTIME}")
print(f"Models dir  : {MODELS_DIR}")


## Step 2 — Explore GPT-2's Tokenizer

GPT-2 uses Byte-level BPE trained on WebText (Reddit links, ~40 GB).
We'll encode support-desk sentences and inspect where it draws token boundaries.


In [ ]:
from transformers import AutoTokenizer

gpt2_tok = AutoTokenizer.from_pretrained("gpt2")

sample_sentences = [
    "I cannot log in to my account.",
    "The application crashes when I upload a PDF.",
    "Please reset my password immediately.",
    "The invoice from last month has an incorrect amount.",
    "I need to cancel my subscription before the renewal date.",
    "Two-factor authentication is not sending the SMS code.",
    "My API key stopped working after the plan upgrade.",
    "The dashboard is showing incorrect usage statistics.",
    "I was charged twice for the same order.",
    "How do I export my data before deleting my account?",
]

print("GPT-2 tokenizer — token boundaries\n")
print(f"{'Sentence':<55} {'Tokens':>6}")
print("-" * 65)
for s in sample_sentences:
    ids = gpt2_tok.encode(s)
    print(f"{s[:54]:<55} {len(ids):>6}")


In [ ]:
# Visualise token boundaries for one sentence
sentence = "I cannot log in to my account."
ids = gpt2_tok.encode(sentence)
tokens = gpt2_tok.convert_ids_to_tokens(ids)

print(f"Sentence : {sentence!r}")
print(f"Token IDs: {ids}")
print(f"Tokens   : {tokens}")
print(f"Count    : {len(ids)}")


## Step 3 — Create a Domain Corpus

We generate 120 synthetic support-desk sentences in Python — no external data download needed.
In a real project this would be your actual ticket history.


In [ ]:
# 120 synthetic support-desk sentences covering common ticket types
DOMAIN_SENTENCES = [
    # Login / account access (20)
    "I cannot log in to my account.",
    "My password reset email never arrived.",
    "Two-factor authentication is blocking my access.",
    "The login page keeps refreshing without signing me in.",
    "I am locked out after too many failed attempts.",
    "SSO login returns a generic error message.",
    "My account was disabled without any notification.",
    "I forgot the email address I used to register.",
    "The remember me option is not working in Chrome.",
    "Session expires after two minutes even with remember me enabled.",
    "Login works on desktop but fails on mobile.",
    "I cannot log in after changing my email address.",
    "Google OAuth is redirecting to an error page.",
    "The CAPTCHA on the login page is not loading.",
    "My account was merged and now I cannot access it.",
    "Biometric login stopped working after the last update.",
    "I receive an invalid credentials error but my password is correct.",
    "The login form does not accept special characters in the password.",
    "My API token is rejected when I try to authenticate.",
    "Account verification link has expired.",
    # Billing / payments (20)
    "I was charged twice for the same subscription.",
    "The invoice from last month has an incorrect amount.",
    "I need a VAT invoice for my company.",
    "My credit card was declined but it has sufficient funds.",
    "I need to update my billing address.",
    "I want to switch from monthly to annual billing.",
    "The coupon code is not applying at checkout.",
    "I requested a refund two weeks ago and have not received it.",
    "I cannot download my past invoices from the billing portal.",
    "The payment confirmation email never arrived.",
    "My subscription renewed even though I cancelled it.",
    "I am being charged for a plan I never selected.",
    "The price shown at checkout does not match the advertised price.",
    "I need to add a secondary payment method.",
    "My bank is flagging your charges as suspicious.",
    "I want to cancel my subscription immediately.",
    "How long does a refund take to appear on my statement?",
    "I received a chargeback dispute email from your team.",
    "I need a receipt for my expense report.",
    "The billing cycle does not match my expected renewal date.",
    # Feature / functionality (20)
    "The export to CSV button is not working.",
    "Dark mode is not saving my preference.",
    "The search filter is returning incorrect results.",
    "Notifications are not appearing even though they are enabled.",
    "The drag and drop interface is unresponsive.",
    "Bulk actions are only applying to the first page of results.",
    "The date picker does not allow me to select past dates.",
    "File upload fails for files larger than 10 MB.",
    "The API endpoint returns a 500 error intermittently.",
    "Sorting by date is ordering results incorrectly.",
    "The mobile app does not sync changes from the web version.",
    "Sharing a report link shows a permission denied error.",
    "The email integration stopped importing messages last Tuesday.",
    "Webhook events are arriving with a 30 minute delay.",
    "The print layout cuts off the right column.",
    "Markdown formatting is not rendering in the preview.",
    "Tags added to items disappear after a page refresh.",
    "The import wizard is stuck on step two of four.",
    "Real-time collaboration shows wrong cursor positions.",
    "The archive function is deleting records instead of archiving.",
    # Performance / reliability (20)
    "The application is extremely slow after the last update.",
    "The dashboard takes over 30 seconds to load.",
    "I am experiencing frequent session timeouts.",
    "The app crashes when I try to open a project with many items.",
    "Images in the editor take a very long time to upload.",
    "The report generation has been running for an hour.",
    "The service has been down for the past two hours.",
    "I am seeing a 502 Bad Gateway error on all pages.",
    "Memory usage on the desktop app grows until it crashes.",
    "The API rate limit is too low for our use case.",
    "Background sync is draining the mobile battery.",
    "The status page shows all systems operational but nothing works.",
    "Caching seems broken because stale data is showing.",
    "The app freezes when I switch between large projects.",
    "Audio in video calls cuts out every few minutes.",
    "Search indexing seems behind by several days.",
    "Third-party integrations are timing out.",
    "The CDN is serving outdated assets after my changes.",
    "Performance degrades significantly after midnight UTC.",
    "The database migration caused data inconsistencies.",
    # Onboarding / setup (20)
    "I am not sure how to set up my first project.",
    "The getting started wizard is missing from my dashboard.",
    "I need help connecting my GitHub repository.",
    "The Slack integration setup returns an OAuth error.",
    "I cannot find where to configure email notifications.",
    "The API documentation does not match the actual response format.",
    "I need to migrate data from our previous tool.",
    "How do I invite my team members to the workspace?",
    "Role permissions are not behaving as documented.",
    "I accidentally deleted the onboarding checklist.",
    "The setup guide links to a page that no longer exists.",
    "I need to configure a custom domain for our account.",
    "SAML SSO configuration is returning a metadata error.",
    "I cannot find the option to enable audit logging.",
    "The trial has expired before I finished evaluating the product.",
    "I need to set up a staging environment separate from production.",
    "The Zapier integration is not triggering on new events.",
    "I am trying to configure IP allowlisting for our organisation.",
    "How do I set up automated backups?",
    "The getting started email series stopped after day one.",
    # Data / privacy (20)
    "I need to export all my data before closing my account.",
    "Please delete my personal data under GDPR.",
    "I received a data breach notification and want to know what was exposed.",
    "I need a copy of the data processing agreement.",
    "Where is my data stored geographically?",
    "I want to know what data you collect about me.",
    "I need to restrict data processing for my account.",
    "My exported data file is corrupted and cannot be opened.",
    "I want to opt out of analytics tracking.",
    "Data I deleted three months ago is still appearing in exports.",
    "I need confirmation that my account has been fully deleted.",
    "How long do you retain support ticket history?",
    "I need to add a data processing addendum to my contract.",
    "My data appears to have been mixed with another customer's data.",
    "I want to review the third parties you share data with.",
    "I need an audit log of all access to my account.",
    "I want to restrict access to specific IP addresses.",
    "Can I host the application on my own infrastructure?",
    "I need a security questionnaire completed for our procurement process.",
    "I want to know if you are SOC 2 compliant.",
]

print(f"Total sentences: {len(DOMAIN_SENTENCES)}")
print("\nSample sentences:")
for s in DOMAIN_SENTENCES[:5]:
    print(f"  {s}")


In [ ]:
import pathlib, tempfile

# Write corpus to a temp file for the tokenizer trainer
corpus_path = PROJECT_ROOT / "data_tokenizer_corpus.txt"
corpus_path.write_text("\n".join(DOMAIN_SENTENCES), encoding="utf-8")
print(f"Corpus written to: {corpus_path}")
print(f"Total characters : {corpus_path.stat().st_size:,}")


## Step 4 — Train a Domain Byte-level BPE Tokenizer

We use HuggingFace `tokenizers` to train a small vocabulary (3,000 tokens) on our corpus.
Small vocab for speed; in production you'd use 8,000–32,000.


In [ ]:
from tokenizers import ByteLevelBPETokenizer

domain_tok = ByteLevelBPETokenizer()

domain_tok.train(
    files=[str(corpus_path)],
    vocab_size=3_000,
    min_frequency=1,
    special_tokens=["<pad>", "<bos>", "<eos>", "<unk>"],
)

domain_tok.save_model(str(MODELS_DIR))
print(f"Tokenizer saved to: {MODELS_DIR}")
print(f"Files: {list(MODELS_DIR.iterdir())}")


In [ ]:
# Verify: reload from disk and encode a test sentence
from tokenizers import ByteLevelBPETokenizer

reloaded_tok = ByteLevelBPETokenizer(
    str(MODELS_DIR / "vocab.json"),
    str(MODELS_DIR / "merges.txt"),
)

test = "I cannot log in to my account."
enc = reloaded_tok.encode(test)
print(f"Test sentence : {test!r}")
print(f"Tokens (domain): {enc.tokens}")
print(f"Token count    : {len(enc.tokens)}")


## Step 5 — Compare GPT-2 vs Domain BPE

Same sentences, two tokenizers. We expect the domain tokenizer to use fewer tokens
on domain text because it learned that support-desk vocabulary is frequent.


In [ ]:
comparison_sentences = sample_sentences  # the 10 from Step 2

rows = []
for s in comparison_sentences:
    gpt2_count = len(gpt2_tok.encode(s))
    domain_count = len(reloaded_tok.encode(s).tokens)
    rows.append({
        "sentence": s[:55],
        "gpt2": gpt2_count,
        "domain": domain_count,
        "delta": gpt2_count - domain_count,
    })

print(f"{'Sentence':<57} {'GPT-2':>6} {'Domain':>7} {'Delta':>6}")
print("-" * 80)
for r in rows:
    print(f"{r['sentence']:<57} {r['gpt2']:>6} {r['domain']:>7} {r['delta']:>+6}")

avg_gpt2 = sum(r["gpt2"] for r in rows) / len(rows)
avg_domain = sum(r["domain"] for r in rows) / len(rows)
print("-" * 80)
print(f"{'Average':<57} {avg_gpt2:>6.1f} {avg_domain:>7.1f} {avg_gpt2-avg_domain:>+6.1f}")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100

labels = [f"S{i+1}" for i in range(len(rows))]
gpt2_counts = [r["gpt2"] for r in rows]
domain_counts = [r["domain"] for r in rows]

x = range(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar([i - width/2 for i in x], gpt2_counts, width, label="GPT-2 tokenizer", color="#4C72B0")
bars2 = ax.bar([i + width/2 for i in x], domain_counts, width, label="Domain BPE (3k vocab)", color="#DD8452")

ax.set_xlabel("Sentence")
ax.set_ylabel("Token count")
ax.set_title("Token count per sentence: GPT-2 vs Domain BPE")
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.legend()
ax.axhline(avg_gpt2, color="#4C72B0", linestyle="--", alpha=0.5, label=f"GPT-2 avg ({avg_gpt2:.1f})")
ax.axhline(avg_domain, color="#DD8452", linestyle="--", alpha=0.5, label=f"Domain avg ({avg_domain:.1f})")
plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / "tokenizer_comparison.png"), bbox_inches="tight")
plt.show()
print("Chart saved.")


In [ ]:
# Key observations
savings = avg_gpt2 - avg_domain
savings_pct = 100 * savings / avg_gpt2

print("=== Key Observations ===")
print(f"GPT-2 average tokens per sentence : {avg_gpt2:.1f}")
print(f"Domain BPE average tokens         : {avg_domain:.1f}")
print(f"Token savings                     : {savings:.1f} ({savings_pct:.0f}%)")
print()
print("Why the domain tokenizer may use MORE or FEWER tokens:")
print("  MORE   — vocab is only 3k (tiny). Rare words split more aggressively.")
print("  FEWER  — frequent domain phrases learned as single tokens.")
print("  In production with vocab_size=8000+ on a real ticket corpus,")
print("  domain savings are consistently 10-20% on in-domain text.")


## Checkpoint

> **Why can the same sentence cost very different token counts across models?**

Write your answer in the cell below. A strong answer covers:
- How BPE merges are learned from a specific corpus
- Why frequency in training determines what gets its own token
- How a general-corpus tokenizer vs a domain-corpus tokenizer diverge on the same text
- Why this matters for cost and context budget


In [ ]:
# Your answer here — this is for your own notes, not executed
answer = """
[Write your answer here]
"""
print(answer)


## Deliverable Summary

After running this notebook you should have:

| Artifact | Location |
|---|---|
| Trained vocabulary | `models/tokenizer/vocab.json` |
| BPE merge rules | `models/tokenizer/merges.txt` |
| Comparison chart | `tokenizer_comparison.png` |

**Next:** Module 1.1 — Data, batching, and the training loop.
You will convert raw text to token IDs using a tokenizer, write the batching logic, and train a bigram model — the simplest possible neural language model.
